In [1]:
import data.breathe_data as bd
import data.helpers as dh
import src.models.helpers as mh
from plotly.subplots import make_subplots
import datetime
import data.cfr_data_19_23 as cfrd
import pandas as pd
import cfr.cfr_viz_helpers as vh
import plotly.express as px
import plotly.graph_objs as go
from scipy.stats import spearmanr, permutation_test

In [2]:
df = bd.load_meas_from_excel(
    # "infer_all_19_data_with_best_FEV1",
    "ppfev1st_ft_bFEV1_2016-19_IV_2019-21_assoc",
    study_folder="CFR",
    str_cols_to_arrays=[
        # "Airway resistance (%)",
        # "P(HFEV1|FEF2575, bFEV1, FEV1)",
        "P(HFEV1|bFEV1)",
        "P(HFEV1|FEV1)",
    ],
    use_csv=True,
    bypass_sanity_checks=True,
)

# Viz ranked associations with IV days

In [26]:
diff_col = "ppFEV1FT - ppFEV1ST"
prctile = 50  # keep rows where abs(diff_col) > this percentile threshold

df_ranked = df.copy()
threshold = df_ranked[diff_col].abs().quantile(prctile / 100)
df_ranked = df_ranked[df_ranked[diff_col].abs() > threshold]

df_ranked.loc[df_ranked["FEV1%PredST"] >= 70, "severity"] = "mild"
df_ranked.loc[(df_ranked["FEV1%PredST"] >= 40) & (df_ranked["FEV1%PredST"] < 70), "severity"] = "moderate"
df_ranked.loc[df_ranked["FEV1%PredST"] < 40, "severity"] = "severe"

iv_max = df_ranked["IV days"].max() * 1.05

severities = [("mild", 1, 2), ("moderate", 3, 4), ("severe", 5, 6)]
fev_metrics = ["FEV1%PredST", "FEV1%PredFT"]
fev_colors = {"FEV1%PredST": "blue", "FEV1%PredFT": "red"}
iv_color = "rgba(64, 64, 64, 0.8)"

# Compute shared y range per severity across both fev_metrics
fev_range_by_severity = {}
for severity_label, _, _ in severities:
    df_sev = df_ranked[df_ranked["severity"] == severity_label]
    vals = pd.concat([df_sev[m] for m in fev_metrics]).dropna()
    pad = (vals.max() - vals.min()) * 0.05
    fev_range_by_severity[severity_label] = [vals.min() - pad, vals.max() + pad]


def add_ranked_fev_iv_panels(fig, df_severity, fev_metric, severity_label, row_fev, row_iv, col):
    df_sorted = df_severity.sort_values(fev_metric, ascending=False).reset_index(drop=True)
    x_rank = list(range(len(df_sorted)))

    fig.add_trace(
        go.Scatter(
            x=x_rank, y=df_sorted[fev_metric],
            mode="markers",
            marker=dict(size=3, opacity=1.0, color=fev_colors[fev_metric]),
            customdata=df_sorted["ID"],
            hovertemplate="ID: %{customdata}<br>" + fev_metric + ": %{y:.1f}<extra></extra>",
            showlegend=False,
        ),
        row=row_fev, col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=x_rank, y=df_sorted["IV days"],
            mode="markers",
            marker=dict(size=3, color=iv_color),
            customdata=df_sorted["ID"],
            hovertemplate="ID: %{customdata}<br>IV days: %{y:.0f}<extra></extra>",
            showlegend=False,
        ),
        row=row_iv, col=col,
    )
    fig.update_xaxes(showticklabels=False, linecolor="black", linewidth=1, showline=True, row=row_fev, col=col)
    fig.update_xaxes(showticklabels=False, linecolor="black", linewidth=1, showline=True, row=row_iv, col=col)
    fig.update_yaxes(title_text=f"{fev_metric}<br>{severity_label} CF", linecolor="black", linewidth=1, showline=True, range=fev_range_by_severity[severity_label], row=row_fev, col=col)
    fig.update_yaxes(title_text="IV days", range=[-5, iv_max], linecolor="black", linewidth=1, showline=True, row=row_iv, col=col)


row_titles = [label for sev, _, _ in severities for label in (sev, "IV days")]

fig = make_subplots(
    rows=6, cols=2,
    vertical_spacing=0.03,
    horizontal_spacing=0.12,
    row_titles=row_titles,
    column_titles=fev_metrics,
)

for col_idx, fev_metric in enumerate(fev_metrics, start=1):
    for severity_label, row_fev, row_iv in severities:
        df_sev = df_ranked[df_ranked["severity"] == severity_label]
        add_ranked_fev_iv_panels(fig, df_sev, fev_metric, severity_label, row_fev, row_iv, col_idx)

fig.update_layout(
    height=1100, width=900,
    title=f"Ranked FEV1 and IV days by severity (|{diff_col}| > {prctile}th pctile)",
    template="simple_white",
    plot_bgcolor="white",
    paper_bgcolor="white",
)
fig.show()


In [14]:
t

0.457945218826417

In [16]:
df_ranked.shape

(1018, 31)

In [13]:
df_ranked[df_ranked["severity"] == "mild"][df_ranked["ID"] == "B161034"]

/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_78140/2327449510.py:1: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.



,ID,Age,Height,FEV1,FEF2575,best FEV1,Sex,Date Recorded,ecFEV1,ecFEF2575,...,bFEV1 % diff,idx best FEV1 2016-19,P(HFEV1|FEV1),P(HFEV1|bFEV1),FEV1%PredST,FEV1%PredFT,ppFEV1FT - ppFEV1ST,IVs,IV days,severity
1504,B161034,24,168,3.9,4.02,3.9,Male,2019-01-01,3.9,4.02,...,1.282049,79,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",89.816357,89.220216,-0.596141,9.0,102.5,mild


# Viz baseline-prediction diff vs IV days

In [107]:
diff_col = "ppFEV1FT - ppFEV1ST"

dftmp = df.copy()

# Filter percentile of certain population
prctile = 90
for prctile in [50, 60, 70, 80, 90]:
    t = dftmp[diff_col].abs().quantile(prctile / 100)
    print(f"{prctile}th percentile of absolute difference: {t}")

    dftmp = dftmp[dftmp[diff_col].abs() > t]

    # Filter by severity level
    dftmp.loc[dftmp["FEV1%PredST"] >= 70, "severity"] = "mild"
    dftmp.loc[(dftmp["FEV1%PredST"] >= 40) & (dftmp["FEV1%PredST"] < 70), "severity"] = (
        "moderate"
    )
    dftmp.loc[dftmp["FEV1%PredST"] < 40, "severity"] = "severe"

    fig = make_subplots(rows=3, cols=1, shared_xaxes=True)

    for i, severity in enumerate(["mild", "moderate", "severe"], start=1):
        dftmp_severity = dftmp[dftmp["severity"] == severity]
        fig.add_trace(
            go.Scatter(
                x=dftmp_severity[diff_col], y=dftmp_severity["IV days"], mode="markers", marker=dict(size=3, opacity=0.8)
            ),
            row=i,
            col=1,
        )
        fig.update_yaxes(title="IV days", range=[-5, dftmp["IV days"].max()*1.1], row=i, col=1)
    fig.update_xaxes(title=f"{diff_col}", row=3, col=1)
        

    title = f"bFEV1_2016_19_IVdays_2019-21_{prctile}th_prctile"
    fig.update_layout(
        height=600, width=800, title=title,
    )

    fig.write_image(dh.get_path_to_main() + f"PlotsCFR/Viz diff vs IV days/{title}.pdf")

##########################################################################################
    def labels_from_bins(bins):
        labels = [f"< {bins[1]}"]
        for i in range(1, len(bins) - 2):
            labels.append(f"[{bins[i]}, {bins[i+1]})")
        labels.append(f">= {bins[-2]}")
        return labels


    fig = make_subplots(3, 1)

    row = 0
    for severity in ["mild", "moderate", "severe"]:
        row += 1
        dftmp2 = dftmp[dftmp["severity"] == severity].copy()
        bins = [-1000, -20, -11, -9, -7, -5, -3, -1, 0]
        labels = labels_from_bins(bins)
        dftmp2["diff_bin"] = pd.cut(dftmp2[diff_col], bins=bins, labels=labels)

        grouped = (
            dftmp2.groupby("diff_bin", observed=False)
            .agg(
                iv_mean=("IV days", "mean"),
                iv_std=("IV days", "std"),
                diff_mean=(diff_col, "mean"),
                count=(diff_col, "size"),
            )
            .reset_index()
        )

        fig.add_trace(
            go.Bar(
                x=grouped["diff_bin"].astype(str),
                y=grouped["iv_mean"],
                # mode="markers+text",
                error_y=dict(type="data", array=grouped["iv_std"].tolist(), visible=True),
                text=grouped["count"].astype(int),
                # textposition="top center",
                name=severity,
            ),
            row=row,
            col=1,
        )
        fig.update_yaxes(title="IV days (mean ± SD)", row=row, col=1)
    fig.update_xaxes(title=f"{diff_col} binned", row=1, col=1)


    title = f"bFEV1_2016_19_IVdays_2019-21_binned_{prctile}th_prctile"

    fig.update_layout(
        xaxis_title=diff_col,
        title=title,
        height=800,
    )
    fig.write_image(dh.get_path_to_main() + f"PlotsCFR/Viz diff vs IV days/{title}.pdf")

50th percentile of absolute difference: 0.457945218826417
60th percentile of absolute difference: 2.781535403334735
70th percentile of absolute difference: 5.978852663315695
80th percentile of absolute difference: 12.69587666933772
90th percentile of absolute difference: 25.269813601145305
